# Notebook 4 - the Validation 

In [1]:
#NB4 - Cleanlab validation of the watchlist
#Independent check: does confident-learning flag the same PLR as the hand-built watchlist?
import numpy as np
import pandas as pd
from cleanlab.filter import find_label_issues

/Users/lauragies/Documents/Job/NF_Bootcamp/capstone-kiezkeeper/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#XGBoost OOF probabilities (all 527 PLR) — the watchlist's basis
oof = pd.read_csv("../../models/M_oof_xgboost.csv", dtype={"plr_id": str})
oof["plr_id"] = oof["plr_id"].astype(str).str.zfill(8)

#the hand-built watchlist (25 PLR) — kept separate, used only for the overlap comparison
watchlist = pd.read_csv("../../models/M2_watchlist.csv", dtype={"plr_id": str})
watchlist["plr_id"] = watchlist["plr_id"].astype(str).str.zfill(8)

print(f"{len(oof)} PLR total | {int(oof['y_true'].sum())} designated | "
      f"{len(watchlist)} on hand-built watchlist")

527 PLR total | 109 designated | 32 on hand-built watchlist


In [3]:
#Cleanlab needs given labels + out-of-sample pred_probs as an (n, n_classes) matrix
#Class 0 = undesignated, Class 1 = designated; our oof_prob is P(class 1)
labels = oof["y_true"].to_numpy().astype(int)
pred_probs = np.column_stack([1 - oof["oof_prob"], oof["oof_prob"]])  # shape (527, 2)

#find_label_issues: PLR whose given label is inconsistent with the model's confident prediction
#ranked by likelihood of being an issue (most suspicious first)
issue_idx = find_label_issues(
    labels=labels,
    pred_probs=pred_probs,
    return_indices_ranked_by="self_confidence",
)

#assemble the flagged PLR with their info
cl_issues = oof.iloc[issue_idx].copy()
cl_issues["cl_rank"] = np.arange(1, len(cl_issues) + 1)

#we care about issues among UNDESIGNATED PLR (label 0 flagged as looking designated)
#= the confident-learning analog of our early-warning watchlist
cl_undes = cl_issues[cl_issues["y_true"] == 0].copy()

print(f"Cleanlab flagged {len(cl_issues)} label issues total")
print(f"  of which {len(cl_undes)} are undesignated PLR (early-warning analog)")
print(f"  and {len(cl_issues) - len(cl_undes)} are designated PLR flagged as not-designated-looking")

Cleanlab flagged 64 label issues total
  of which 38 are undesignated PLR (early-warning analog)
  and 26 are designated PLR flagged as not-designated-looking


In [4]:
#Overlap: how many of the 25 hand-built watchlist PLR does Cleanlab independently flag?
wl_ids = set(watchlist["plr_id"])
cl_undes_ids = set(cl_undes["plr_id"])

overlap = wl_ids & cl_undes_ids
only_watchlist = wl_ids - cl_undes_ids
only_cleanlab  = cl_undes_ids - wl_ids

print(f"Hand-built watchlist:      {len(wl_ids)} PLR")
print(f"Cleanlab undesignated flags: {len(cl_undes_ids)} PLR")
print(f"Overlap (both agree):        {len(overlap)} PLR "
      f"({len(overlap)/len(wl_ids):.0%} of the watchlist)")
print(f"Only on hand-built watchlist: {len(only_watchlist)}")
print(f"Only flagged by Cleanlab:     {len(only_cleanlab)}")

#which watchlist PLR did Cleanlab NOT flag? (worth inspecting)
if only_watchlist:
    print("\nOn watchlist but not flagged by Cleanlab:")
    print(watchlist[watchlist["plr_id"].isin(only_watchlist)]
          [["plr_name", "bez", "oof_prob", "status"]].to_string(index=False))

Hand-built watchlist:      32 PLR
Cleanlab undesignated flags: 38 PLR
Overlap (both agree):        32 PLR (100% of the watchlist)
Only on hand-built watchlist: 0
Only flagged by Cleanlab:     6


## Result

All 32 of the watchlist's planning areas are independently flagged by Cleanlab (100% overlap). Cleanlab, using a completely different method (label-quality/confident-learning on the training labels, not the oof_prob ranking) points at every single area that the watchlist surfaces. That's strong convergent validation: two independent signals agree that these 32 undesignated PLR look designation-like.

In [6]:
#which 6 PLR does Cleanlab flag beyond the hand-built 32? (likely ranks 26+)
extra = cl_undes[cl_undes["plr_id"].isin(only_cleanlab)].copy()
extra = extra.merge(
    pd.read_csv("../../data/final_datasets/dataset_0.csv", dtype={"plr_id": str})
      .assign(plr_id=lambda d: d["plr_id"].astype(str).str.zfill(8))[["plr_id", "plr_name", "bez"]],
    on="plr_id", how="left"
)
print(extra[["plr_id", "plr_name", "bez", "oof_prob"]]
      .sort_values("oof_prob", ascending=False).to_string(index=False))

  plr_id           plr_name                             bez  oof_prob
12100204         Schäfersee              12 - Reinickendorf  0.694774
08100101         Hasenheide                   08 - Neukölln  0.658168
04501151    Hildegardstraße 04 - Charlottenburg-Wilmersdorf  0.638546
04501149 Nikolsburger Platz 04 - Charlottenburg-Wilmersdorf  0.616660
01200624        Heidestraße                      01 - Mitte  0.604541
04501043        Preußenpark 04 - Charlottenburg-Wilmersdorf  0.586272


In [ ]:
what it could mean that cleanlab selected 436 additional planning areas from the weakest fold: 

It could mean the weak fold is under-flagged by your oof_prob ranking — the model's sparse-fold scores are depressed, those three CW PLR score 0.59–0.64 and miss the cutoff, but Cleanlab (working from label quality, not the fold-limited probabilities) still catches them. If so, the weak fold is costing you real candidates, and these three are evidence of it. That's a genuine finding worth a sentence.
Or it could mean Cleanlab is picking up the same label-sparsity noise in that district and flagging on thin evidence. Cleanlab's confident-learning is only as good as the label signal, and CW is where the label signal is weakest.

--> Cleanlab surfaces 6 further candidates just below the watchlist cutoff (oof_prob 0.59–0.69); notably three are in Charlottenburg-Wilmersdorf, the weak-labelling fold, consistent with that district's known score suppression.
                                                                            
                                                                            The Charlottenburg-Wilmersdorf point stands and is arguably sharper now: three of the six are in the weak fold, scoring 0.586–0.639 — well below both your boundaries. Given the fold suppresses oof_prob there, it's plausible their "true" similarity is higher and the fold is what keeps them out. That's the finding worth stating: Cleanlab, not being bound by the fold-limited probabilities, flags them anyway.

### Result Discussion - umschreiben

Cleanlab, applied to the same XGBoost out-of-fold probabilities via confident learning, independently flags all 25 hand-built watchlist PLR as label issues — a 100% overlap. This confirms the watchlist is not an artefact of the chosen rank cutoff: a formal, established method identifies exactly the same core set.

Cleanlab additionally flags 13 further undesignated PLR, all with oof_prob between 0.59 and 0.77 — i.e. just below the N=25 threshold (0.824) — which is consistent with the deliberately conservative cutoff placed at the last visible gap in the probability ranking; below it the probabilities decline smoothly with no natural boundary. 

The two methods therefore agree completely on the core watchlist and differ only at the margin, where the choice of boundary is inherently a judgment call rather than a data-driven one. One caveat: because both approaches rest on the same out-of-fold probabilities, Cleanlab validates the selection rule, not the underlying probabilities — it is a strong consistency check, not a fully independent second source.

In [8]:
# --- interpret the 6 Cleanlab-only PLR: near-miss vs. weak-fold-only ----------

# bring in milieu_anteil (needed for urgency + the urgency-entrant floor)
labels_full = pd.read_csv("../../data/Module 2/target_milieuschutz.csv", dtype={"plr_id": str})
labels_full["plr_id"] = labels_full["plr_id"].astype(str).str.zfill(8)

# full undesignated set with both scores, ranked exactly as NB3 did
und = oof[oof["y_true"] == 0].merge(
    labels_full[["plr_id", "milieu_anteil"]], on="plr_id", how="left"
)
und["urgency"] = und["oof_prob"] * (1 - und["milieu_anteil"])
und["rank_oof"]     = und["oof_prob"].rank(ascending=False, method="min").astype(int)
und["rank_urgency"] = und["urgency"].rank(ascending=False, method="min").astype(int)

# weak-fold flag (strip the "NN - " prefix first)
und["bez_name"] = und["bez"].str.split(" - ").str[-1].str.strip() if "bez" in und.columns else None

N_WATCHLIST = 25
urg_floor = und.loc[und["rank_urgency"] <= N_WATCHLIST, "oof_prob"].min()
print(f"oof top-25 cutoff:          ~0.77 (NB3 cell 4)")
print(f"urgency-entrant oof floor:  {urg_floor:.3f}")
print(f"=> union watchlist floor is the LOWER of the two\n")

# look up the 6 Cleanlab-only PLR with their full-set ranks
look = und[und["plr_id"].isin(only_cleanlab)].copy()

# bez/plr_name aren't on oof — pull them from dataset_0 like the cell above does
meta = (pd.read_csv("../../data/final_datasets/dataset_0.csv", dtype={"plr_id": str})
          .assign(plr_id=lambda d: d["plr_id"].astype(str).str.zfill(8))
          [["plr_id", "plr_name", "bez"]])
look = look.merge(meta, on="plr_id", how="left")
look["bez_name"] = look["bez"].str.split(" - ").str[-1].str.strip()
look["weak_fold"] = look["bez_name"].isin(["Charlottenburg-Wilmersdorf", "Lichtenberg"])

look = look.sort_values("oof_prob", ascending=False)
print(look[["plr_name", "bez_name", "oof_prob", "milieu_anteil",
            "rank_oof", "rank_urgency", "weak_fold"]].to_string(index=False))

oof top-25 cutoff:          ~0.77 (NB3 cell 4)
urgency-entrant oof floor:  0.690
=> union watchlist floor is the LOWER of the two

          plr_name                   bez_name  oof_prob  milieu_anteil  rank_oof  rank_urgency  weak_fold
        Schäfersee              Reinickendorf  0.694774       0.152368        31            30      False
        Hasenheide                   Neukölln  0.658168       0.133260        34            33      False
   Hildegardstraße Charlottenburg-Wilmersdorf  0.638546       0.000000        35            26       True
Nikolsburger Platz Charlottenburg-Wilmersdorf  0.616660       0.000000        36            28       True
       Heidestraße                      Mitte  0.604541       0.037452        37            32      False
       Preußenpark Charlottenburg-Wilmersdorf  0.586272       0.000523        38            31       True


These six aren't "surfaced only by Cleanlab because the weak fold hid them." They're the immediate next tier on both the orderings: rank_oof 31–38, and for the zero-coverage CW ones, rank_urgency 26–31. Cleanlab, the oof ranking, and the urgency ranking all three agree they're the strongest candidates just past the line. The three methods draw the boundary within a handful of ranks of each other. 

This is perceived as as strong convergent-validation rather than "the fold hid candidates". The question is rather where to draw the line than what the models says.
